<span style="color:gray; font-size:smaller;">Written by Salil Deshpande</span>

This tutorial is all about the MotifCompendium object. It focuses on the various options you have for building a MotifCompendium object and about the flexible syntax you have for manipulating, modifying, and accessing internal data from MotifCompendium objects.

Let's get started!

We will begin by importing the packages we will need.

In [1]:
import gdown
import numpy as np
import pandas as pd

import MotifCompendium
import MotifCompendium.utils.loader as utils_loader
import MotifCompendium.utils.motif as utils_motif

This tutorial will be written as if we had access to a 12GB GPU. If you want to run this tutorial yourself but do not have access to a GPU then set `use_gpu=False` and do not set the `max_chunk` parameter.

In [2]:
MotifCompendium.set_compute_options(max_cpus=4, use_gpu=True, max_chunk=1152, progress_bar=True)

We will also download all the data that we will need for this tutorial.

In [3]:
cardiomyocyte_file_id = "1iAeA2dczPsDEhDp3d4qsVt0oBtU9X07R"
cardiomyocyte_file_url = f"https://drive.google.com/uc?id={cardiomyocyte_file_id}"
cardiomyocyte_output_file = "tutorial_data/cardiomyocyte_modisco_output.h5"
gdown.download(cardiomyocyte_file_url, cardiomyocyte_output_file, quiet=False)

endothelial_file_id = "1Na2-QIa5davufwc_R4Awa87MQf6zBhR4"
endothelial_file_url = f"https://drive.google.com/uc?id={endothelial_file_id}"
endothelial_output_file = "tutorial_data/endothelial_modisco_output.h5"
gdown.download(endothelial_file_url, endothelial_output_file, quiet=False)

h13_file_id = "1Zrpxjo89LtRBVzOIHmg9HpnX0cEGGmUB"
h13_file_url = f"https://drive.google.com/uc?id={h13_file_id}"
h13_pfm_file = "tutorial_data/H13CORE_pfms.txt"
gdown.download(h13_file_url, h13_pfm_file, quiet=False)

viestra_file_id = "1bMXV9eDIqFi9dKZizWTj8EQSfK7x-CKR"
viestra_file_url = f"https://drive.google.com/uc?id={viestra_file_id}"
viestra_meme_file = "tutorial_data/viestra_v21beta_all_dbs.meme"
gdown.download(viestra_file_url, viestra_meme_file, quiet=False)

tutorial_mc_id = "1XQ7pWbEeP2RKEg5d5jJ1JOjotXXPFaeL"
tutorial_mc_url = f"https://drive.google.com/uc?id={tutorial_mc_id}"
tutorial_mc_file = "tutorial_data/tutorial_motif_compendium.mc"
gdown.download(tutorial_mc_url, tutorial_mc_file, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1iAeA2dczPsDEhDp3d4qsVt0oBtU9X07R
From (redirected): https://drive.google.com/uc?id=1iAeA2dczPsDEhDp3d4qsVt0oBtU9X07R&confirm=t&uuid=3968ec71-1847-4ab0-ae33-9cbd1ccdaa3d
To: /users/salil512/experiments/MotifCompendium/tutorials/tutorial_data/cardiomyocyte_modisco_output.h5
100%|██████████| 484M/484M [00:04<00:00, 102MB/s]  
Downloading...
From (original): https://drive.google.com/uc?id=1Na2-QIa5davufwc_R4Awa87MQf6zBhR4
From (redirected): https://drive.google.com/uc?id=1Na2-QIa5davufwc_R4Awa87MQf6zBhR4&confirm=t&uuid=f0a868a2-375b-42b4-91c4-c0e5c7054578
To: /users/salil512/experiments/MotifCompendium/tutorials/tutorial_data/endothelial_modisco_output.h5
100%|██████████| 241M/241M [00:02<00:00, 97.5MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1Zrpxjo89LtRBVzOIHmg9HpnX0cEGGmUB
To: /users/salil512/experiments/MotifCompendium/tutorials/tutorial_data/H13CORE_pfms.txt
100%|██████████| 1.62M/1.62M [00:00<00:00, 6.72MB

'tutorial_data/tutorial_motif_compendium.mc'

# Part 1 - Building MotifCompendium Objects

In the first tutorial, we saw saw how a MotifCompendium object could be built from Modisco output files.

In [4]:
modisco_dict = {
    "cardiomyocyte": "tutorial_data/cardiomyocyte_modisco_output.h5",
    "endothelial": "tutorial_data/endothelial_modisco_output.h5"
}
mc = MotifCompendium.build_from_modisco(modisco_dict)
mc

computing similarities on GPU...: 100%|██████████| 1/1 [00:03<00:00,  3.02s/it]


,name,num_seqlets,model,posneg,avg_contrib,avg_dist_from_summit
0,cardiomyocyte-pos.pattern_0,24730,cardiomyocyte,pos,0.629150,15.721553
1,cardiomyocyte-pos.pattern_1,23251,cardiomyocyte,pos,0.421469,33.662853
2,cardiomyocyte-pos.pattern_10,4122,cardiomyocyte,pos,0.383973,33.419699
3,cardiomyocyte-pos.pattern_11,4008,cardiomyocyte,pos,0.351602,36.965818
4,cardiomyocyte-pos.pattern_12,3352,cardiomyocyte,pos,0.453516,42.597554
...,...,...,...,...,...,...
83,endothelial-pos.pattern_5,3338,endothelial,pos,0.479641,37.803475
84,endothelial-pos.pattern_6,2977,endothelial,pos,0.437019,35.074236
85,endothelial-pos.pattern_7,2358,endothelial,pos,0.469403,37.488550
86,endothelial-pos.pattern_8,2115,endothelial,pos,0.529135,51.907329


In general, you can build a MotifCompendium from one of three ways: from Modicso files, from PFM files, or with motifs directly. We will now discuss the options available with those three approaches in more detail.

The `build_from_modisco()` function can take a few optional arguments.

You can set the `load_subpatterns` argument to `True` to load motifs at the subpattern level so that each motif in the resulting MotifCompendium is a Modisco subpattern instead of the Modisco subpattern.

In [5]:
mc_subpattern = MotifCompendium.build_from_modisco(modisco_dict, load_subpatterns=True)
mc_subpattern

computing similarities on GPU...: 100%|██████████| 1/1 [00:00<00:00, 11.81it/s]


,name,num_seqlets,model,posneg,avg_contrib,avg_dist_from_summit
0,cardiomyocyte-pos.pattern_0-subpattern_0,4174,cardiomyocyte,pos,0.615092,18.702923
1,cardiomyocyte-pos.pattern_0-subpattern_1,3778,cardiomyocyte,pos,0.645625,14.714664
2,cardiomyocyte-pos.pattern_0-subpattern_2,3594,cardiomyocyte,pos,0.656586,14.798275
3,cardiomyocyte-pos.pattern_0-subpattern_3,2784,cardiomyocyte,pos,0.641359,15.193606
4,cardiomyocyte-pos.pattern_0-subpattern_4,2681,cardiomyocyte,pos,0.695037,16.481910
...,...,...,...,...,...,...
774,endothelial-pos.pattern_9-subpattern_5,146,endothelial,pos,0.446366,41.876712
775,endothelial-pos.pattern_9-subpattern_6,123,endothelial,pos,0.511323,41.024390
776,endothelial-pos.pattern_9-subpattern_7,100,endothelial,pos,0.485628,37.720000
777,endothelial-pos.pattern_9-subpattern_8,80,endothelial,pos,0.422155,38.525000


Notice how there are now 779 motifs instead of 88 because each Modisco pattern can have many subpatterns.

The optional `modisco_region_width` argument should match the `--window/-w` option used when generating the Modisco objects. By default, this number will be 400 (which matches the tfmodisco-lite default). **Don't change this argument unless you specifically know you are dealing with a non-standard Modisco window size.**

And the `ic` argument is a boolean that indicates whether or not motifs should be information-content scaled when being ingested from Modisco. By default, `ic` is set to `True`. IC-scaling is very helpful for the performance of downstream motif clustering, so **it is recommended to never change this option without good reason.** If you would like to disable IC-scaling, however, do it by setting `ic=False` as shown below.

In [6]:
mc_no_ic = MotifCompendium.build_from_modisco(modisco_dict, ic=False)
mc_no_ic

computing similarities on GPU...: 100%|██████████| 1/1 [00:00<00:00, 88.60it/s]


,name,num_seqlets,model,posneg,avg_contrib,avg_dist_from_summit
0,cardiomyocyte-pos.pattern_0,24730,cardiomyocyte,pos,0.629150,15.721553
1,cardiomyocyte-pos.pattern_1,23251,cardiomyocyte,pos,0.421469,33.662853
2,cardiomyocyte-pos.pattern_10,4122,cardiomyocyte,pos,0.383973,33.419699
3,cardiomyocyte-pos.pattern_11,4008,cardiomyocyte,pos,0.351602,36.965818
4,cardiomyocyte-pos.pattern_12,3352,cardiomyocyte,pos,0.453516,42.597554
...,...,...,...,...,...,...
83,endothelial-pos.pattern_5,3338,endothelial,pos,0.479641,37.803475
84,endothelial-pos.pattern_6,2977,endothelial,pos,0.437019,35.074236
85,endothelial-pos.pattern_7,2358,endothelial,pos,0.469403,37.488550
86,endothelial-pos.pattern_8,2115,endothelial,pos,0.529135,51.907329


The `build_from_pfm()` function allows you to build a MotifCompendium directly from files containing PFMs. The function supports files in the PFM or MEME formats. Similarly to `build_from_modisco()`, pass in a dictionary of motif set name to file path.

In [7]:
mc_pfm = MotifCompendium.build_from_pfm({"h13": "tutorial_data/H13CORE_pfms.txt", "viestra": "tutorial_data/viestra_v21beta_all_dbs.meme"})
mc_pfm

computing similarities on GPU...: 100%|██████████| 36/36 [00:06<00:00,  5.63it/s]
/users/salil512/experiments/MotifCompendium/MotifCompendium/MotifCompendium.py:577: RuntimeWarning: self.alignment_rc is not symmetric. This may be due to numerical instability (especially if you have very symmetric motifs).
  warnings.warn(
/users/salil512/experiments/MotifCompendium/MotifCompendium/MotifCompendium.py:597: RuntimeWarning: self.alignment_h should be symmetric for reverse complement motifs and skew-symmetric for motifs that are already aligned, but it is not. This is very rare and should not occur (unless you are working with PFMs, for which it happens often).
  warnings.warn(


,name,posneg,source
0,h13-AHCTF1.H13CORE.0.B.B,pos,h13
1,h13-AHR.H13CORE.0.P.B,pos,h13
2,h13-AHRR.H13CORE.0.P.C,pos,h13
3,h13-ALX1.H13CORE.0.SM.B,pos,h13
4,h13-ALX3.H13CORE.0.SM.B,pos,h13
...,...,...,...
6906,viestra-MA1984.1,pos,viestra
6907,viestra-MA1985.1,pos,viestra
6908,viestra-MA1986.1,pos,viestra
6909,viestra-MA1987.1,pos,viestra


The `ic` argument is also an optional argument for `build_from_pfm()`. Setting it to True indicates that the motifs should be information-content scaled when being loaded. If you are working with PFMs, the IC-scaling will transform them into PWMs.  By default, `ic` is set to `True`, but if you specifically want to work with PFMs instead of PWMs or you are already loading in PWMs, you may want ot set `ic=False`, as shown below.

In [8]:
mc_pfm_no_ic = MotifCompendium.build_from_pfm({"h13": "tutorial_data/H13CORE_pfms.txt", "viestra": "tutorial_data/viestra_v21beta_all_dbs.meme"}, ic=False)
mc_pfm_no_ic

computing similarities on GPU...: 100%|██████████| 36/36 [00:06<00:00,  5.97it/s]
/users/salil512/experiments/MotifCompendium/MotifCompendium/MotifCompendium.py:577: RuntimeWarning: self.alignment_rc is not symmetric. This may be due to numerical instability (especially if you have very symmetric motifs).
  warnings.warn(
/users/salil512/experiments/MotifCompendium/MotifCompendium/MotifCompendium.py:597: RuntimeWarning: self.alignment_h should be symmetric for reverse complement motifs and skew-symmetric for motifs that are already aligned, but it is not. This is very rare and should not occur (unless you are working with PFMs, for which it happens often).
  warnings.warn(


,name,posneg,source
0,h13-AHCTF1.H13CORE.0.B.B,pos,h13
1,h13-AHR.H13CORE.0.P.B,pos,h13
2,h13-AHRR.H13CORE.0.P.C,pos,h13
3,h13-ALX1.H13CORE.0.SM.B,pos,h13
4,h13-ALX3.H13CORE.0.SM.B,pos,h13
...,...,...,...
6906,viestra-MA1984.1,pos,viestra
6907,viestra-MA1985.1,pos,viestra
6908,viestra-MA1986.1,pos,viestra
6909,viestra-MA1987.1,pos,viestra


Both `build_from_modisco()` and `build_from_pfm()` load motifs by calling functions from the loader submodule. You can call these directly yourself if you'd like to load motifs on your own.

`build_from_modisco()` calls the `load_modiscos()` function, which takes in the exact Modisco file path dictionary you pass to `build_from_modisco()`. `load_modiscos()` returns motifs learned by Modisco as well as the names of the motifs as per Modisco, the number of seqlets that make up each motif, the model from which the motifs came from, a classification of each motif by Modisco as either containing a positive motif or a negative motif, and the average distance of the constituent seqlets from peak summits, and the average contribution value of the constituent seqlets.

In [9]:
modisco_motifs, motif_names, seqlet_counts, model_names, posneg, avgdist_summits, avg_contribs = utils_loader.load_modiscos(modisco_dict)
modisco_metadata = pd.DataFrame({"name": motif_names, "num_seqlets": seqlet_counts, "model_names": model_names, "posneg": posneg, "avg_dist_from_summit": avgdist_summits, "avg_contrib": avg_contribs})
print(modisco_motifs.shape)
modisco_metadata

(88, 30, 4)


,name,num_seqlets,model_names,posneg,avg_dist_from_summit,avg_contrib
0,cardiomyocyte-pos.pattern_0,24730,cardiomyocyte,pos,15.721553,0.629150
1,cardiomyocyte-pos.pattern_1,23251,cardiomyocyte,pos,33.662853,0.421469
2,cardiomyocyte-pos.pattern_10,4122,cardiomyocyte,pos,33.419699,0.383973
3,cardiomyocyte-pos.pattern_11,4008,cardiomyocyte,pos,36.965818,0.351602
4,cardiomyocyte-pos.pattern_12,3352,cardiomyocyte,pos,42.597554,0.453516
...,...,...,...,...,...,...
83,endothelial-pos.pattern_5,3338,endothelial,pos,37.803475,0.479641
84,endothelial-pos.pattern_6,2977,endothelial,pos,35.074236,0.437019
85,endothelial-pos.pattern_7,2358,endothelial,pos,37.488550,0.469403
86,endothelial-pos.pattern_8,2115,endothelial,pos,51.907329,0.529135


If you would like to load from a single Modisco file, you may call the `load_modisco()` function. Pass it the path to a single Modisco output file and it will return the same information as before without the model name for each motif.

In [10]:
modisco_motifs, motif_names, seqlet_counts, posneg, avgdist_summits, avg_contribs = utils_loader.load_modisco("tutorial_data/cardiomyocyte_modisco_output.h5")
single_modisco_metadata = pd.DataFrame({"name": motif_names, "num_seqlets": seqlet_counts, "posneg": posneg, "avg_dist_from_summit": avgdist_summits, "avg_contrib": avg_contribs})
print(modisco_motifs.shape)
single_modisco_metadata

(45, 30, 4)


,name,num_seqlets,posneg,avg_dist_from_summit,avg_contrib
0,pos.pattern_0,24730,pos,15.721553,0.629150
1,pos.pattern_1,23251,pos,33.662853,0.421469
2,pos.pattern_10,4122,pos,33.419699,0.383973
3,pos.pattern_11,4008,pos,36.965818,0.351602
4,pos.pattern_12,3352,pos,42.597554,0.453516
5,pos.pattern_13,3313,pos,35.456686,0.330949
6,pos.pattern_14,2975,pos,37.126387,0.403911
7,pos.pattern_15,2457,pos,42.288563,0.517382
8,pos.pattern_16,2131,pos,32.916471,0.393503
9,pos.pattern_17,1742,pos,45.091848,0.567991


Both `load_modiscos()` and `load_modisco()` can also take the optional arguments `load_subpatterns`, `modisco_region_width`, and `ic`, which were described earlier.

Similarly, `build_from_pfm()` calls the `load_pfms()` function, which takes in the exact PFM file path dictionary you pass to `build_from_pfm()`. `load_pfms()` returns motifs in the PFM files, the names for each motif, and the files from which they came. `load_pfms()` can evan take a mix of files in PFM and MEME format.

In [11]:
pfm_motifs, motif_names, pfm_files = utils_loader.load_pfms({"h13": "tutorial_data/H13CORE_pfms.txt", "viestra": "tutorial_data/viestra_v21beta_all_dbs.meme"})
pfm_metadata = pd.DataFrame({"name": motif_names, "source": pfm_files})
print(pfm_motifs.shape)
pfm_metadata

(6911, 39, 4)


,name,source
0,h13-AHCTF1.H13CORE.0.B.B,h13
1,h13-AHR.H13CORE.0.P.B,h13
2,h13-AHRR.H13CORE.0.P.C,h13
3,h13-ALX1.H13CORE.0.SM.B,h13
4,h13-ALX3.H13CORE.0.SM.B,h13
...,...,...
6906,viestra-MA1984.1,viestra
6907,viestra-MA1985.1,viestra
6908,viestra-MA1986.1,viestra
6909,viestra-MA1987.1,viestra


If you would like to load from a single PFM file, you may call the `load_pfm()` function. Pass it the path to a single file containing PFMs and it will return the motifs in that file and their names.

In [12]:
pfm_motifs, motif_names = utils_loader.load_pfm("tutorial_data/H13CORE_pfms.txt")
single_pfm_metadata = pd.DataFrame({"name": motif_names})
print(pfm_motifs.shape)
single_pfm_metadata

(1610, 31, 4)


,name
0,AHCTF1.H13CORE.0.B.B
1,AHR.H13CORE.0.P.B
2,AHRR.H13CORE.0.P.C
3,ALX1.H13CORE.0.SM.B
4,ALX3.H13CORE.0.SM.B
...,...
1605,ZSCAN12.H13CORE.0.P.B
1606,ZSCAN2.H13CORE.0.PG.A
1607,ZSCAN25.H13CORE.0.PSG.A
1608,ZXDA.H13CORE.0.PSI.A


Both `'load_pfms()` and `load_pfm()` can take the optional `ic` argument, which was described earlier.

Both the `load_modicsos()` and `load_pfms()` functions can parallelize their file loading if the compute option `max_cpus` is set to higher than 1.

Under the hood, both `build_from_modisco()` and `build_from_pfm()` load motifs by calling the appropriate functions from the loader submodule and then calling the `build()` function. The `build()` function allows you to build a MotifCompendium by passing in motifs directly. The motifs must be passed in as a *non-negative*, *normalized* **motif stack**. We define a *motif stack* as an `np.ndarray` of shape `(N, L, C)` where the first dimension is the number of motifs, the second dimension is the length of the motifs, and the third dimension is the number of bases/channels. Here, non-negativity means that each of the `N` motifs of shape `(L, C)` must be >= 0, and normalized means that their sum is 1. Written programmatically, these conditions are that `np.all(motifs >= 0) and np.all(np.sum(motifs, axis=(1, 2)) == 1)`.

The loader functions return normalized motif stacks. Below, let's see how we can pass PFMs that we load directly into `build()`.

In [13]:
pfm_motifs, motif_names, pfm_files = utils_loader.load_pfms({"h13": "tutorial_data/H13CORE_pfms.txt", "viestra": "tutorial_data/viestra_v21beta_all_dbs.meme"})
mc_pfm = MotifCompendium.build(pfm_motifs)
mc_pfm

computing similarities on GPU...: 100%|██████████| 36/36 [00:03<00:00, 10.00it/s]
/users/salil512/experiments/MotifCompendium/MotifCompendium/MotifCompendium.py:577: RuntimeWarning: self.alignment_rc is not symmetric. This may be due to numerical instability (especially if you have very symmetric motifs).
  warnings.warn(
/users/salil512/experiments/MotifCompendium/MotifCompendium/MotifCompendium.py:597: RuntimeWarning: self.alignment_h should be symmetric for reverse complement motifs and skew-symmetric for motifs that are already aligned, but it is not. This is very rare and should not occur (unless you are working with PFMs, for which it happens often).
  warnings.warn(


,name
0,motif_0
1,motif_1
2,motif_2
3,motif_3
4,motif_4
...,...
6906,motif_6906
6907,motif_6907
6908,motif_6908
6909,motif_6909


The `build()` function can also be passed a `metadata` argument as a `pd.DataFrame`. This will populate the per-motif metadata that you see when you print a MotifCompendium object. Let's see that usage below:

In [14]:
pfm_motifs, motif_names, pfm_files = utils_loader.load_pfms({"h13": "tutorial_data/H13CORE_pfms.txt", "viestra": "tutorial_data/viestra_v21beta_all_dbs.meme"})
pfm_metadata = pd.DataFrame({"name": motif_names, "source": pfm_files})
mc_pfm = MotifCompendium.build(pfm_motifs, pfm_metadata)
mc_pfm

computing similarities on GPU...: 100%|██████████| 36/36 [00:03<00:00, 10.12it/s]
/users/salil512/experiments/MotifCompendium/MotifCompendium/MotifCompendium.py:577: RuntimeWarning: self.alignment_rc is not symmetric. This may be due to numerical instability (especially if you have very symmetric motifs).
  warnings.warn(
/users/salil512/experiments/MotifCompendium/MotifCompendium/MotifCompendium.py:597: RuntimeWarning: self.alignment_h should be symmetric for reverse complement motifs and skew-symmetric for motifs that are already aligned, but it is not. This is very rare and should not occur (unless you are working with PFMs, for which it happens often).
  warnings.warn(


,name,source
0,h13-AHCTF1.H13CORE.0.B.B,h13
1,h13-AHR.H13CORE.0.P.B,h13
2,h13-AHRR.H13CORE.0.P.C,h13
3,h13-ALX1.H13CORE.0.SM.B,h13
4,h13-ALX3.H13CORE.0.SM.B,h13
...,...,...
6906,viestra-MA1984.1,viestra
6907,viestra-MA1985.1,viestra
6908,viestra-MA1986.1,viestra
6909,viestra-MA1987.1,viestra


Traditionally, PFMs and PWMs have been non-negative matrix representations of motifs, so they automatically pass the non-negativity requirement. On the other hand, CWMs (contribution weight matrices) like those in Modisco outputs, are learned as features of computational sequence-to-function models and can contain bases with negative attributions. How do we make sure we don't lose this important positivity/negativity information while still enforcing that we have a non-negative matrix? **The solution is an 8 channel motif representation. Instead of having a 4 channel (A, C, G, T) representation, we can split each base into a positive and negative subchannel in order to have an 8 channel (A+, A-, C+, C-, G-, G+, T-, T+)**.

The 4 channel to 8 channel transformation is performed automatically during `build_from_modisco()`. Below, however, we will see an example of how to perform that transformation manually. First, we will load the 4 channel motifs from Modisco using the `load_modiscos()` function from the loader submodule. Then, we will apply the `motif_4_to_8()` function from the motif submodule to create our 8 channel motifs.

In [15]:
modisco_motifs, motif_names, seqlet_counts, model_names, posneg, avgdist_summits, avg_contribs = utils_loader.load_modiscos(modisco_dict)
modisco_metadata = pd.DataFrame({"name": motif_names, "num_seqlets": seqlet_counts, "model_names": model_names, "posneg": posneg, "avg_dist_from_summit": avgdist_summits, "avg_contrib": avg_contribs})
modisco_motifs_8 = utils_motif.motif_4_to_8(modisco_motifs)
mc = MotifCompendium.build(modisco_motifs_8, modisco_metadata)
mc

computing similarities on GPU...: 100%|██████████| 1/1 [00:00<00:00, 56.56it/s]


,name,num_seqlets,model_names,posneg,avg_dist_from_summit,avg_contrib
0,cardiomyocyte-pos.pattern_0,24730,cardiomyocyte,pos,15.721553,0.629150
1,cardiomyocyte-pos.pattern_1,23251,cardiomyocyte,pos,33.662853,0.421469
2,cardiomyocyte-pos.pattern_10,4122,cardiomyocyte,pos,33.419699,0.383973
3,cardiomyocyte-pos.pattern_11,4008,cardiomyocyte,pos,36.965818,0.351602
4,cardiomyocyte-pos.pattern_12,3352,cardiomyocyte,pos,42.597554,0.453516
...,...,...,...,...,...,...
83,endothelial-pos.pattern_5,3338,endothelial,pos,37.803475,0.479641
84,endothelial-pos.pattern_6,2977,endothelial,pos,35.074236,0.437019
85,endothelial-pos.pattern_7,2358,endothelial,pos,37.488550,0.469403
86,endothelial-pos.pattern_8,2115,endothelial,pos,51.907329,0.529135


Notice how in the display, it says `Motifs = (88, 30, 8)`. The 8 indicates that the motifs are in an 8 channel format. The `build_from_modisco()` automatically transforms the Modisco motifs into an 8 channel format. And, even though it is superfluous in most cases, `build_from_pfm()` does so to its motifs, as well.

**As a strict rule, MotifCompendium only accepts 4 or 8 channel motif representations.**

If you had another random source of un-normalized CWMs, you would have to normalize and bring them to an 8 channel format yourself as shown in the example below.=

In [16]:
random_motifs = np.random.rand(100, 30, 4) - 0.5
random_motifs_8 = utils_motif.motif_4_to_8(random_motifs)
random_motifs_8 /= np.sum(random_motifs_8, axis=(1, 2), keepdims=True)
mc = MotifCompendium.build(random_motifs_8)
mc

computing similarities on GPU...: 100%|██████████| 1/1 [00:00<00:00, 95.74it/s]


,name
0,motif_0
1,motif_1
2,motif_2
3,motif_3
4,motif_4
...,...
95,motif_95
96,motif_96
97,motif_97
98,motif_98


The final thing to know about building MotifCompendium objects is the `safe` option. By default, the `load()`, `build()`, `build_from_modisco()`, and `build_from_pfm()` functions all have an option called `safe`. `safe` is a boolean that determines whether or not object integrity checks are performed at the time of object instantiation. These checks include some hard requirements which will throw an error if something is badly wrong with the structure of the object, as well as other checks that simply warn of possible numerical precision errors. It is important to note that in general, PFMs are much more likely to trigger these numerical warnings than CWMs are. In fact, you may have noticed some of these warnings being given when processing some the PFMs above.

By default, `safe` is set to `True`, but it can be overridden to be `False`. While these checks can help guarantee object integrity, they can be computationally costly, especially for extremely large MotifCompendium. In general, it is best practice to let `safe` be `True` by default. However, if a MotifCompendium object is very large or you use it frequently and don't need to worry about changes in the object integrity, there is no problem in setting `safe` to be `False`.

An example of loading with `safe` turned to `False` is given below.

In [17]:
mc_tutorial = MotifCompendium.load("tutorial_data/tutorial_motif_compendium.mc", safe=False)
mc_tutorial

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster,annotation
0,cardiomyocyte-pos.pattern_0,24730,cardiomyocyte,pos,15.721553,0,CTCF#1_0
1,cardiomyocyte-pos.pattern_1,23251,cardiomyocyte,pos,33.662853,8,NFI#1_8
2,cardiomyocyte-pos.pattern_10,4122,cardiomyocyte,pos,33.419699,11,HD:MEIS-TGIF_11
3,cardiomyocyte-pos.pattern_11,4008,cardiomyocyte,pos,36.965818,12,GATA#1_12
4,cardiomyocyte-pos.pattern_12,3352,cardiomyocyte,pos,42.597554,13,ETS:ELF-ETV#1_13
...,...,...,...,...,...,...,...
83,endothelial-pos.pattern_5,3338,endothelial,pos,37.803475,9,BZIP:FOSL-JUND#1_9
84,endothelial-pos.pattern_6,2977,endothelial,pos,35.074236,10,NFY_10
85,endothelial-pos.pattern_7,2358,endothelial,pos,37.488550,4,BZIP:ATF-CREB#1_4
86,endothelial-pos.pattern_8,2115,endothelial,pos,51.907329,76,ETS:ELF-SPIB#1_76


# Part 2 - Sorting And Slicing MotifCompendium Objects

MotifCompendium offers a familiar and intuitive Pandas-like syntax for sorting and slicing MotifCompendium objects.

To sort a MotifCompendium, simply, call the `.sort()` method with a column you would like to sort on.

In [18]:
mc_tutorial.sort("num_seqlets")

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster,annotation
0,endothelial-pos.pattern_42,21,endothelial,pos,37.571429,75,RFX#1_75
1,cardiomyocyte-neg.pattern_3,22,cardiomyocyte,neg,46.090909,44,ZEB-SNAI_44
2,endothelial-pos.pattern_41,22,endothelial,pos,43.545455,74,SOX#1_74
3,endothelial-pos.pattern_40,23,endothelial,pos,53.347826,73,Unknown_73
4,endothelial-pos.pattern_39,25,endothelial,pos,93.160000,71,HSF1_71
...,...,...,...,...,...,...,...
83,cardiomyocyte-pos.pattern_3,21827,cardiomyocyte,pos,32.562194,27,NFIhalf_27
84,cardiomyocyte-pos.pattern_2,22917,cardiomyocyte,pos,43.456255,2,MEF2_2
85,cardiomyocyte-pos.pattern_1,23251,cardiomyocyte,pos,33.662853,8,NFI#1_8
86,cardiomyocyte-pos.pattern_0,24730,cardiomyocyte,pos,15.721553,0,CTCF#1_0


The direction of sorting can be changed by setting the value of the `ascending` argument to True or False. By default, it is `True` and sorts from smallest to largest.

In [19]:
mc_tutorial.sort("num_seqlets", ascending=False)

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster,annotation
0,endothelial-pos.pattern_0,24805,endothelial,pos,45.475912,45,ETS:ELF-ETV#1_45
1,cardiomyocyte-pos.pattern_0,24730,cardiomyocyte,pos,15.721553,0,CTCF#1_0
2,cardiomyocyte-pos.pattern_1,23251,cardiomyocyte,pos,33.662853,8,NFI#1_8
3,cardiomyocyte-pos.pattern_2,22917,cardiomyocyte,pos,43.456255,2,MEF2_2
4,cardiomyocyte-pos.pattern_3,21827,cardiomyocyte,pos,32.562194,27,NFIhalf_27
...,...,...,...,...,...,...,...
83,endothelial-pos.pattern_39,25,endothelial,pos,93.160000,71,HSF1_71
84,endothelial-pos.pattern_40,23,endothelial,pos,53.347826,73,Unknown_73
85,cardiomyocyte-neg.pattern_3,22,cardiomyocyte,neg,46.090909,44,ZEB-SNAI_44
86,endothelial-pos.pattern_41,22,endothelial,pos,43.545455,74,SOX#1_74


MotifCompendium objects can be sorted by multiple columns when a list of column names is passed in instead of a single column name. If this is done, the ascending argument can be passed a list of boolean values corresponding to whether or not each column should be sorted in ascending order.

In [20]:
mc_tutorial.sort(["posneg", "num_seqlets"], ascending=[True, False])

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster,annotation
0,cardiomyocyte-neg.pattern_0,55,cardiomyocyte,neg,114.309091,41,NFYrepressive_41
1,cardiomyocyte-neg.pattern_1,50,cardiomyocyte,neg,81.700000,42,YY1-2repressive_42
2,cardiomyocyte-neg.pattern_2,46,cardiomyocyte,neg,53.630435,43,ZBT7A.H13CORE.0.P.B_43
3,cardiomyocyte-neg.pattern_3,22,cardiomyocyte,neg,46.090909,44,ZEB-SNAI_44
4,endothelial-pos.pattern_0,24805,endothelial,pos,45.475912,45,ETS:ELF-ETV#1_45
...,...,...,...,...,...,...,...
83,cardiomyocyte-pos.pattern_40,27,cardiomyocyte,pos,29.444444,39,HD:MEIS-TGIF_39
84,endothelial-pos.pattern_39,25,endothelial,pos,93.160000,71,HSF1_71
85,endothelial-pos.pattern_40,23,endothelial,pos,53.347826,73,Unknown_73
86,endothelial-pos.pattern_41,22,endothelial,pos,43.545455,74,SOX#1_74


While `sort()` returned a sorted MotifCompendium object, notice how the original MotifCompendium did not change.

In [21]:
mc_tutorial

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster,annotation
0,cardiomyocyte-pos.pattern_0,24730,cardiomyocyte,pos,15.721553,0,CTCF#1_0
1,cardiomyocyte-pos.pattern_1,23251,cardiomyocyte,pos,33.662853,8,NFI#1_8
2,cardiomyocyte-pos.pattern_10,4122,cardiomyocyte,pos,33.419699,11,HD:MEIS-TGIF_11
3,cardiomyocyte-pos.pattern_11,4008,cardiomyocyte,pos,36.965818,12,GATA#1_12
4,cardiomyocyte-pos.pattern_12,3352,cardiomyocyte,pos,42.597554,13,ETS:ELF-ETV#1_13
...,...,...,...,...,...,...,...
83,endothelial-pos.pattern_5,3338,endothelial,pos,37.803475,9,BZIP:FOSL-JUND#1_9
84,endothelial-pos.pattern_6,2977,endothelial,pos,35.074236,10,NFY_10
85,endothelial-pos.pattern_7,2358,endothelial,pos,37.488550,4,BZIP:ATF-CREB#1_4
86,endothelial-pos.pattern_8,2115,endothelial,pos,51.907329,76,ETS:ELF-SPIB#1_76


If you'd like, the current MotifCompendium can be sorted in-place by setting the `inplace` argument to `True`.

In [22]:
mc_tutorial.sort(["posneg", "num_seqlets"], ascending=[True, False], inplace=True)
mc_tutorial

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster,annotation
0,cardiomyocyte-neg.pattern_0,55,cardiomyocyte,neg,114.309091,41,NFYrepressive_41
1,cardiomyocyte-neg.pattern_1,50,cardiomyocyte,neg,81.700000,42,YY1-2repressive_42
2,cardiomyocyte-neg.pattern_2,46,cardiomyocyte,neg,53.630435,43,ZBT7A.H13CORE.0.P.B_43
3,cardiomyocyte-neg.pattern_3,22,cardiomyocyte,neg,46.090909,44,ZEB-SNAI_44
4,endothelial-pos.pattern_0,24805,endothelial,pos,45.475912,45,ETS:ELF-ETV#1_45
...,...,...,...,...,...,...,...
83,cardiomyocyte-pos.pattern_40,27,cardiomyocyte,pos,29.444444,39,HD:MEIS-TGIF_39
84,endothelial-pos.pattern_39,25,endothelial,pos,93.160000,71,HSF1_71
85,endothelial-pos.pattern_40,23,endothelial,pos,53.347826,73,Unknown_73
86,endothelial-pos.pattern_41,22,endothelial,pos,43.545455,74,SOX#1_74


New columns can easily be added to MotifCompendium objects using the same column assignment syntax as Pandas. The examples below show how a single value of a list can be assigned to a new MotifCompendium column.

In [23]:
mc_tutorial["test_column_1"] = 1
mc_tutorial["test_column_2"] = list(range(len(mc_tutorial)))
mc_tutorial

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster,annotation,test_column_1,test_column_2
0,cardiomyocyte-neg.pattern_0,55,cardiomyocyte,neg,114.309091,41,NFYrepressive_41,1,0
1,cardiomyocyte-neg.pattern_1,50,cardiomyocyte,neg,81.700000,42,YY1-2repressive_42,1,1
2,cardiomyocyte-neg.pattern_2,46,cardiomyocyte,neg,53.630435,43,ZBT7A.H13CORE.0.P.B_43,1,2
3,cardiomyocyte-neg.pattern_3,22,cardiomyocyte,neg,46.090909,44,ZEB-SNAI_44,1,3
4,endothelial-pos.pattern_0,24805,endothelial,pos,45.475912,45,ETS:ELF-ETV#1_45,1,4
...,...,...,...,...,...,...,...,...,...
83,cardiomyocyte-pos.pattern_40,27,cardiomyocyte,pos,29.444444,39,HD:MEIS-TGIF_39,1,83
84,endothelial-pos.pattern_39,25,endothelial,pos,93.160000,71,HSF1_71,1,84
85,endothelial-pos.pattern_40,23,endothelial,pos,53.347826,73,Unknown_73,1,85
86,endothelial-pos.pattern_41,22,endothelial,pos,43.545455,74,SOX#1_74,1,86


MotifCompendium objects can also be indexed by strings corresponding to column names. Doing so will return a `pd.Series` corresponding to that column in the MotifCompendium's metadata.

In [24]:
mc_tutorial["name"]

0      cardiomyocyte-neg.pattern_0
1      cardiomyocyte-neg.pattern_1
2      cardiomyocyte-neg.pattern_2
3      cardiomyocyte-neg.pattern_3
4        endothelial-pos.pattern_0
                  ...             
83    cardiomyocyte-pos.pattern_40
84      endothelial-pos.pattern_39
85      endothelial-pos.pattern_40
86      endothelial-pos.pattern_41
87      endothelial-pos.pattern_42
Name: name, Length: 88, dtype: object

MotifCompendium objects can also be indexed by boolean `pd.Series`. Doing so will return a sliced MotifCompendium object corresponding only to motifs for which the indexing `pd.Series` was `True`. This allows for a very flexible syntax for getting MotifCompendium objects corresponding to motifs you are interested in.

In [25]:
mc_ctcf = mc_tutorial[mc_tutorial["annotation"] == "CTCF#1_0"]
mc_ctcf

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster,annotation,test_column_1,test_column_2
0,cardiomyocyte-pos.pattern_0,24730,cardiomyocyte,pos,15.721553,0,CTCF#1_0,1,5
1,endothelial-pos.pattern_1,13746,endothelial,pos,16.345264,0,CTCF#1_0,1,9


Here are a few more examples of performing slices in this sort of way.

In [26]:
mc_many_seqlets = mc_tutorial[mc_tutorial["num_seqlets"] >= 10000]
mc_many_seqlets

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster,annotation,test_column_1,test_column_2
0,endothelial-pos.pattern_0,24805,endothelial,pos,45.475912,45,ETS:ELF-ETV#1_45,1,4
1,cardiomyocyte-pos.pattern_0,24730,cardiomyocyte,pos,15.721553,0,CTCF#1_0,1,5
2,cardiomyocyte-pos.pattern_1,23251,cardiomyocyte,pos,33.662853,8,NFI#1_8,1,6
3,cardiomyocyte-pos.pattern_2,22917,cardiomyocyte,pos,43.456255,2,MEF2_2,1,7
4,cardiomyocyte-pos.pattern_3,21827,cardiomyocyte,pos,32.562194,27,NFIhalf_27,1,8
5,endothelial-pos.pattern_1,13746,endothelial,pos,16.345264,0,CTCF#1_0,1,9


In [27]:
mc_no_positive_or_endothelial = mc_tutorial[~((mc_tutorial["posneg"] == "pos") & (mc_tutorial["model"] == "endothelial"))]
mc_no_positive_or_endothelial

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster,annotation,test_column_1,test_column_2
0,cardiomyocyte-neg.pattern_0,55,cardiomyocyte,neg,114.309091,41,NFYrepressive_41,1,0
1,cardiomyocyte-neg.pattern_1,50,cardiomyocyte,neg,81.700000,42,YY1-2repressive_42,1,1
2,cardiomyocyte-neg.pattern_2,46,cardiomyocyte,neg,53.630435,43,ZBT7A.H13CORE.0.P.B_43,1,2
3,cardiomyocyte-neg.pattern_3,22,cardiomyocyte,neg,46.090909,44,ZEB-SNAI_44,1,3
4,cardiomyocyte-pos.pattern_0,24730,cardiomyocyte,pos,15.721553,0,CTCF#1_0,1,5
5,cardiomyocyte-pos.pattern_1,23251,cardiomyocyte,pos,33.662853,8,NFI#1_8,1,6
6,cardiomyocyte-pos.pattern_2,22917,cardiomyocyte,pos,43.456255,2,MEF2_2,1,7
7,cardiomyocyte-pos.pattern_3,21827,cardiomyocyte,pos,32.562194,27,NFIhalf_27,1,8
8,cardiomyocyte-pos.pattern_4,9499,cardiomyocyte,pos,42.382672,38,MEF2_38,1,10
9,cardiomyocyte-pos.pattern_5,8321,cardiomyocyte,pos,36.823699,6,SP-KLF_6,1,11


# Part 3 - Accessing Values Of Other MotifCompendium Attributes

In this section, we will discuss how to interact with the various data and attributes of MotifCompendium objects.

## Motifs

The motifs are the core of any MotifCompendium object. They can be accessed directly with `.motifs`. The motifs are a `np.ndarray` of shape `(N, L, C)` where `N` is the number of motifs, `L` is the length of the motifs, and `C` are the number of "nucleotide channels" in the motif representation.

In [28]:
motifs = mc_tutorial.motifs
print(type(motifs), motifs.shape)

<class 'numpy.ndarray'> (88, 30, 8)


The motifs in MotifCompendium objects can have 4 or 8 channels. Even when motifs are represented as 8 channel motifs internally to MotifCompendium, users may care more about the corresponding 4 channel motifs. No matter if your motifs are 4 or 8 channels, you can always get the `(N, L, C)` 4 channel motif representations using the `get_standard_motif_stack()` method. **This is the recommended way to access your motifs.**

In [29]:
motifs = mc_tutorial.get_standard_motif_stack()
print(type(motifs), motifs.shape)

<class 'numpy.ndarray'> (88, 30, 4)


The channels in the motif are always arranged in such a way that the motifs can always be reverse-complemented using the same syntax.

In [30]:
motifs8 = mc_tutorial.motifs
motifs8_revcomp = motifs8[:, ::-1, ::-1]
print(type(motifs8_revcomp), motifs8_revcomp.shape)
motifs4 = mc_tutorial.get_standard_motif_stack()
motifs4_revcomp = motifs4[:, ::-1, ::-1]
print(type(motifs4_revcomp), motifs4_revcomp.shape)

<class 'numpy.ndarray'> (88, 30, 8)
<class 'numpy.ndarray'> (88, 30, 4)


Alternatively, there is a `reverse_complement()` function in the motif submodule that we recommend using.

In [31]:
motifs8 = mc_tutorial.motifs
motifs8_revcomp = utils_motif.reverse_complement(motifs8)
print(type(motifs8_revcomp), motifs8_revcomp.shape)
motifs4 = mc_tutorial.get_standard_motif_stack()
motifs4_revcomp = utils_motif.reverse_complement(motifs4)
print(type(motifs4_revcomp), motifs4_revcomp.shape)

<class 'numpy.ndarray'> (88, 30, 8)
<class 'numpy.ndarray'> (88, 30, 4)


## Similarity & Alignment

Let's discuss the concepts of similarity and alignment as they are used in MotifCompendium.

**Alignment** refers to the way in which two motifs can be arranged with respect to one another. There are two attributes needed to specify how one motif should be aligned with another motif. First, you must specify if one motif should be reverse complemented to best match the other motif. Once this is done, you may need to specify a horizontal shift needed to best align the two motifs.


For any particular alignment between two motifs, one can compute a measure of how similar those motifs are. In MotifCompendium, we measure this **similarity** as a number between 0 and 1. At a given alignment, a score of 1 can be achieved iff two motifs are exactly equal at that alignment. However, the similarity at each alignment is not meaningful, as low similarities can be achieved for every motif pair by simply shifting them until there is no overlap between them. So, for any two motifs, their pairwise similarity is considered to be the maximum similarity across all of their possible alignments.

The MotifCompendium objects store information about pairwise motif similarity and alignment in the `.similarity`, `.alignment_h`, and `.alignment_rc` attributes. Each one of them is a square `np.ndarray` of shape `(N, N)` where `N` is the number of motifs in the MotifCompendium. Here is a little more information about each of them.

- `.similarity` - `similarity[i, j]` is the similarity between the i<sup>th</sup> and j<sup>th</sup> motifs. This is the maximum similarity between the two motifs across all possible alignments. `similarity` is symmetric. The diagonal of `similarity` is all 1s because each motif has full similarity with itself.
- `.alignment_h` - `alignment_h[i, j]` is an integer representing how many positions to the right the j<sup>th</sup> motif must be shifted (after being reverse complemented, if needed) in order to achieve the best possible alignment with the i<sup>th</sup> motif.
- `.alignment_rc` - `alignment_rc[i, j]` is a boolean representing whether or not the j<sup>th</sup> motif must be reverse complemented in order to achieve the best possible alignment with the i<sup>th</sup> motif.

Here is an example of how one would access the similarity and alignment information between motifs.

In [32]:
print(f"similarity: {mc_tutorial.similarity[0, 1]}")
print(f"horizontal alignment: {mc_tutorial.alignment_h[0, 1]}")
print(f"reverse complement: {mc_tutorial.alignment_rc[0, 1]}")

similarity: 0.6910644173622131
horizontal alignment: 4
reverse complement: False


The above means that the 0<sup>th</sup> and 1<sup>st</sup> motifs have a similarity of 0.691, and that their maximum similarity is achieved when the 1<sup>st</sup> motif is reverse complemented and *then* shifted 4 positions to the right.

And here is an example showing a motif's similarity with itself. Notice how the similarity is 1, the horizontal alignment is 0, and it does not need to be reverse complemented to match with itself.

In [33]:
print(f"similarity: {mc_tutorial.similarity[2, 2]}")
print(f"horizontal alignment: {mc_tutorial.alignment_h[2, 2]}")
print(f"reverse complement: {mc_tutorial.alignment_rc[2, 2]}")

similarity: 1.0
horizontal alignment: 0
reverse complement: False


In almost all cases, you will never need to access alignment information yourself. MotifCompendium will take care of alignment information as needed behind the scenes.

Sometimes, you may want to access similarity information by indexing in a more interpretable manner. The `.get_similarity_slice()` method can help you access slices of the similarity matrix by passing in conditions corresponding to certain motifs.

Let's suppose I wanted to look at the similarity of negative motifs with respect to all other motifs. That can be done by passing in a `pd.Series` of booleans corresponding to all the negative motifs. As shown in the slicing section, this can easily be done in a Pandas-like syntax.

In [34]:
endothelial_motif_similarity_slice = mc_tutorial.get_similarity_slice(mc_tutorial["model"] == "endothelial")
endothelial_motif_similarity_slice

,0,1,2,3,4,5,6,7,8,9,...,78,79,80,81,82,83,84,85,86,87
4,0.002069,0.002909,0.006847,0.023028,1.000000,0.464071,0.627148,0.243060,0.655419,0.459783,...,0.500559,0.233582,0.478073,0.823286,0.594629,0.473325,0.453168,0.866266,0.335923,0.324169
9,0.004936,0.004012,0.007935,0.018227,0.459783,0.997925,0.497928,0.296337,0.583620,1.000000,...,0.315990,0.262084,0.359594,0.397281,0.429860,0.531467,0.447451,0.418498,0.340419,0.322643
13,0.001784,0.002621,0.008329,0.023084,0.634150,0.613345,0.620388,0.163096,0.665269,0.618650,...,0.306175,0.136444,0.352055,0.470664,0.539143,0.416494,0.404661,0.576625,0.297929,0.396823
14,0.003845,0.003338,0.007322,0.022204,0.597372,0.470578,0.994201,0.247427,0.769246,0.460548,...,0.308462,0.219096,0.336547,0.457787,0.855556,0.523977,0.413681,0.621136,0.428505,0.348036
20,0.001768,0.001888,0.004340,0.016386,0.358523,0.366372,0.415788,0.459399,0.500495,0.357574,...,0.438831,0.433410,0.431859,0.601638,0.287275,0.483702,0.330033,0.313800,0.934189,0.412497
22,0.001416,0.001417,0.004263,0.014604,0.453224,0.287744,0.399587,0.290958,0.517635,0.282567,...,0.422882,0.281116,0.305516,0.545374,0.288990,0.669318,0.354409,0.404029,0.434129,0.314682
24,0.001775,0.002560,0.005095,0.015897,0.468890,0.382783,0.540950,0.466590,0.613295,0.372384,...,0.506286,0.449816,0.323483,0.387819,0.391428,0.437592,0.406620,0.414646,0.725500,0.326138
27,0.001867,0.002482,0.006928,0.020314,0.458259,0.350357,0.377406,0.275476,0.495284,0.363467,...,0.418331,0.256496,0.424200,0.391504,0.355906,0.650716,0.438497,0.374957,0.487856,0.621185
29,0.002082,0.002368,0.007245,0.020410,0.798687,0.549569,0.505956,0.293779,0.539790,0.543698,...,0.390912,0.257444,0.434575,0.665136,0.530455,0.414629,0.371065,0.709734,0.321184,0.355829
30,0.001873,0.002151,0.007766,0.019871,0.450903,0.468623,0.342492,0.163767,0.431924,0.458383,...,0.268650,0.168193,0.358884,0.451741,0.356212,0.344400,0.524196,0.410767,0.322886,0.408607


As shown above, `.get_similarity_slice()` can be passed a `pd.Series` of booleans and will return a `pd.DataFrame` where each row corresponds to the selected motifs. The returned `pd.DataFrame` contains one column for each motif in the original MotifCompendium. If you want to look at the similarity as compared to a specific set of motifs instead of with respect to all motifs in the original MotifCompendium, you can do this by passing in a second `pd.Series` of booleans corresponding to the other motifs you are interested in looking at.

In [35]:
endothelial_vs_cardiomyocyte_motif_similarity_slice = mc_tutorial.get_similarity_slice(mc_tutorial["model"] == "endothelial", mc_tutorial["model"] == "cardiomyocyte")
endothelial_vs_cardiomyocyte_motif_similarity_slice

,0,1,2,3,5,6,7,8,10,11,...,67,68,69,74,75,78,79,80,82,83
4,0.002069,0.002909,0.006847,0.023028,0.464071,0.627148,0.243060,0.655419,0.324759,0.578680,...,0.507759,0.434003,0.236193,0.495518,0.517179,0.500559,0.233582,0.478073,0.594629,0.473325
9,0.004936,0.004012,0.007935,0.018227,0.997925,0.497928,0.296337,0.583620,0.421141,0.620049,...,0.544448,0.344086,0.275293,0.320616,0.343382,0.315990,0.262084,0.359594,0.429860,0.531467
13,0.001784,0.002621,0.008329,0.023084,0.613345,0.620388,0.163096,0.665269,0.306061,0.972886,...,0.871789,0.310779,0.261356,0.515977,0.454779,0.306175,0.136444,0.352055,0.539143,0.416494
14,0.003845,0.003338,0.007322,0.022204,0.470578,0.994201,0.247427,0.769246,0.318078,0.578054,...,0.504067,0.405228,0.316907,0.707157,0.385607,0.308462,0.219096,0.336547,0.855556,0.523977
20,0.001768,0.001888,0.004340,0.016386,0.366372,0.415788,0.459399,0.500495,0.511285,0.260759,...,0.226846,0.427449,0.403796,0.285266,0.434901,0.438831,0.433410,0.431859,0.287275,0.483702
22,0.001416,0.001417,0.004263,0.014604,0.287744,0.399587,0.290958,0.517635,0.329325,0.282904,...,0.251827,0.920261,0.311338,0.310034,0.437929,0.422882,0.281116,0.305516,0.288990,0.669318
24,0.001775,0.002560,0.005095,0.015897,0.382783,0.540950,0.466590,0.613295,0.556785,0.410703,...,0.363781,0.386576,0.398590,0.316426,0.468789,0.506286,0.449816,0.323483,0.391428,0.437592
27,0.001867,0.002482,0.006928,0.020314,0.350357,0.377406,0.275476,0.495284,0.288492,0.511058,...,0.447505,0.536767,0.286410,0.316271,0.596592,0.418331,0.256496,0.424200,0.355906,0.650716
29,0.002082,0.002368,0.007245,0.020410,0.549569,0.505956,0.293779,0.539790,0.344091,0.596927,...,0.540545,0.339643,0.271572,0.388369,0.461664,0.390912,0.257444,0.434575,0.530455,0.414629
30,0.001873,0.002151,0.007766,0.019871,0.468623,0.342492,0.163767,0.431924,0.198300,0.532319,...,0.463070,0.305301,0.210938,0.348294,0.566102,0.268650,0.168193,0.358884,0.356212,0.344400


Notice how this has returned a `pd.DataFrame` with the same rows as before, but now with only a subset of the columns corresponding to the cardiomyocyte motifs.

If you ever want to see the rows and columns labeled with the motif names, you can set the `with_names` parameter to be `True`.

In [36]:
endothelial_vs_cardiomyocyte_motif_similarity_slice = mc_tutorial.get_similarity_slice(mc_tutorial["model"] == "endothelial", mc_tutorial["model"] == "cardiomyocyte", with_names=True)
endothelial_vs_cardiomyocyte_motif_similarity_slice

,cardiomyocyte-neg.pattern_0,cardiomyocyte-neg.pattern_1,cardiomyocyte-neg.pattern_2,cardiomyocyte-neg.pattern_3,cardiomyocyte-pos.pattern_0,cardiomyocyte-pos.pattern_1,cardiomyocyte-pos.pattern_2,cardiomyocyte-pos.pattern_3,cardiomyocyte-pos.pattern_4,cardiomyocyte-pos.pattern_5,...,cardiomyocyte-pos.pattern_31,cardiomyocyte-pos.pattern_32,cardiomyocyte-pos.pattern_33,cardiomyocyte-pos.pattern_34,cardiomyocyte-pos.pattern_35,cardiomyocyte-pos.pattern_36,cardiomyocyte-pos.pattern_37,cardiomyocyte-pos.pattern_38,cardiomyocyte-pos.pattern_39,cardiomyocyte-pos.pattern_40
endothelial-pos.pattern_0,0.002069,0.002909,0.006847,0.023028,0.464071,0.627148,0.243060,0.655419,0.324759,0.578680,...,0.507759,0.434003,0.236193,0.495518,0.517179,0.500559,0.233582,0.478073,0.594629,0.473325
endothelial-pos.pattern_1,0.004936,0.004012,0.007935,0.018227,0.997925,0.497928,0.296337,0.583620,0.421141,0.620049,...,0.544448,0.344086,0.275293,0.320616,0.343382,0.315990,0.262084,0.359594,0.429860,0.531467
endothelial-pos.pattern_2,0.001784,0.002621,0.008329,0.023084,0.613345,0.620388,0.163096,0.665269,0.306061,0.972886,...,0.871789,0.310779,0.261356,0.515977,0.454779,0.306175,0.136444,0.352055,0.539143,0.416494
endothelial-pos.pattern_3,0.003845,0.003338,0.007322,0.022204,0.470578,0.994201,0.247427,0.769246,0.318078,0.578054,...,0.504067,0.405228,0.316907,0.707157,0.385607,0.308462,0.219096,0.336547,0.855556,0.523977
endothelial-pos.pattern_4,0.001768,0.001888,0.004340,0.016386,0.366372,0.415788,0.459399,0.500495,0.511285,0.260759,...,0.226846,0.427449,0.403796,0.285266,0.434901,0.438831,0.433410,0.431859,0.287275,0.483702
endothelial-pos.pattern_5,0.001416,0.001417,0.004263,0.014604,0.287744,0.399587,0.290958,0.517635,0.329325,0.282904,...,0.251827,0.920261,0.311338,0.310034,0.437929,0.422882,0.281116,0.305516,0.288990,0.669318
endothelial-pos.pattern_6,0.001775,0.002560,0.005095,0.015897,0.382783,0.540950,0.466590,0.613295,0.556785,0.410703,...,0.363781,0.386576,0.398590,0.316426,0.468789,0.506286,0.449816,0.323483,0.391428,0.437592
endothelial-pos.pattern_7,0.001867,0.002482,0.006928,0.020314,0.350357,0.377406,0.275476,0.495284,0.288492,0.511058,...,0.447505,0.536767,0.286410,0.316271,0.596592,0.418331,0.256496,0.424200,0.355906,0.650716
endothelial-pos.pattern_8,0.002082,0.002368,0.007245,0.020410,0.549569,0.505956,0.293779,0.539790,0.344091,0.596927,...,0.540545,0.339643,0.271572,0.388369,0.461664,0.390912,0.257444,0.434575,0.530455,0.414629
endothelial-pos.pattern_9,0.001873,0.002151,0.007766,0.019871,0.468623,0.342492,0.163767,0.431924,0.198300,0.532319,...,0.463070,0.305301,0.210938,0.348294,0.566102,0.268650,0.168193,0.358884,0.356212,0.344400


And if you want a `np.ndarray` of results, you can use the `pd.DataFrame` `.to_numpy()` method.

In [37]:
endothelial_vs_cardiomyocyte_motif_similarity_slice_np = mc_tutorial.get_similarity_slice(mc_tutorial["model"] == "endothelial", mc_tutorial["model"] == "cardiomyocyte", with_names=True).to_numpy()
print(type(endothelial_vs_cardiomyocyte_motif_similarity_slice_np), endothelial_vs_cardiomyocyte_motif_similarity_slice_np.shape)

<class 'numpy.ndarray'> (43, 45)


## Metadata

MotifCompendium objects maintain metadata tables in the `.metadata` attribute. The metadata serves as a way to flexibly store information about each motif. `.metadata` is a `pd.DataFrame` and can be accessed directly.

In [38]:
metadata = mc_tutorial.metadata
print(type(metadata))
metadata

<class 'pandas.core.frame.DataFrame'>


,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster,annotation,test_column_1,test_column_2
0,cardiomyocyte-neg.pattern_0,55,cardiomyocyte,neg,114.309091,41,NFYrepressive_41,1,0
1,cardiomyocyte-neg.pattern_1,50,cardiomyocyte,neg,81.700000,42,YY1-2repressive_42,1,1
2,cardiomyocyte-neg.pattern_2,46,cardiomyocyte,neg,53.630435,43,ZBT7A.H13CORE.0.P.B_43,1,2
3,cardiomyocyte-neg.pattern_3,22,cardiomyocyte,neg,46.090909,44,ZEB-SNAI_44,1,3
4,endothelial-pos.pattern_0,24805,endothelial,pos,45.475912,45,ETS:ELF-ETV#1_45,1,4
...,...,...,...,...,...,...,...,...,...
83,cardiomyocyte-pos.pattern_40,27,cardiomyocyte,pos,29.444444,39,HD:MEIS-TGIF_39,1,83
84,endothelial-pos.pattern_39,25,endothelial,pos,93.160000,71,HSF1_71,1,84
85,endothelial-pos.pattern_40,23,endothelial,pos,53.347826,73,Unknown_73,1,85
86,endothelial-pos.pattern_41,22,endothelial,pos,43.545455,74,SOX#1_74,1,86


The columns of the metadata can be gotten directly with the `.columns()` method, which returns a list of column names.

In [39]:
mc_tutorial.columns()

['name',
 'num_seqlets',
 'model',
 'posneg',
 'avg_dist_from_summit',
 'cluster',
 'annotation',
 'test_column_1',
 'test_column_2']

As shown above, the best way to get and set columns of the metadata is with the `[]` indexing operator.

In [40]:
mc_tutorial["name"] # Get metadata column value

0      cardiomyocyte-neg.pattern_0
1      cardiomyocyte-neg.pattern_1
2      cardiomyocyte-neg.pattern_2
3      cardiomyocyte-neg.pattern_3
4        endothelial-pos.pattern_0
                  ...             
83    cardiomyocyte-pos.pattern_40
84      endothelial-pos.pattern_39
85      endothelial-pos.pattern_40
86      endothelial-pos.pattern_41
87      endothelial-pos.pattern_42
Name: name, Length: 88, dtype: object

In [41]:
mc_tutorial["test_column_3"] = "new test column"
mc_tutorial

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster,annotation,test_column_1,test_column_2,test_column_3
0,cardiomyocyte-neg.pattern_0,55,cardiomyocyte,neg,114.309091,41,NFYrepressive_41,1,0,new test column
1,cardiomyocyte-neg.pattern_1,50,cardiomyocyte,neg,81.700000,42,YY1-2repressive_42,1,1,new test column
2,cardiomyocyte-neg.pattern_2,46,cardiomyocyte,neg,53.630435,43,ZBT7A.H13CORE.0.P.B_43,1,2,new test column
3,cardiomyocyte-neg.pattern_3,22,cardiomyocyte,neg,46.090909,44,ZEB-SNAI_44,1,3,new test column
4,endothelial-pos.pattern_0,24805,endothelial,pos,45.475912,45,ETS:ELF-ETV#1_45,1,4,new test column
...,...,...,...,...,...,...,...,...,...,...
83,cardiomyocyte-pos.pattern_40,27,cardiomyocyte,pos,29.444444,39,HD:MEIS-TGIF_39,1,83,new test column
84,endothelial-pos.pattern_39,25,endothelial,pos,93.160000,71,HSF1_71,1,84,new test column
85,endothelial-pos.pattern_40,23,endothelial,pos,53.347826,73,Unknown_73,1,85,new test column
86,endothelial-pos.pattern_41,22,endothelial,pos,43.545455,74,SOX#1_74,1,86,new test column


The best way to rename columns is with the `.rename_columns()` method, which takes in a dictionary of old column names to new column names.

In [42]:
mc_tutorial.rename_columns({"cluster": "cluster_id", "test_column_1": "1_value"})
mc_tutorial

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster_id,annotation,1_value,test_column_2,test_column_3
0,cardiomyocyte-neg.pattern_0,55,cardiomyocyte,neg,114.309091,41,NFYrepressive_41,1,0,new test column
1,cardiomyocyte-neg.pattern_1,50,cardiomyocyte,neg,81.700000,42,YY1-2repressive_42,1,1,new test column
2,cardiomyocyte-neg.pattern_2,46,cardiomyocyte,neg,53.630435,43,ZBT7A.H13CORE.0.P.B_43,1,2,new test column
3,cardiomyocyte-neg.pattern_3,22,cardiomyocyte,neg,46.090909,44,ZEB-SNAI_44,1,3,new test column
4,endothelial-pos.pattern_0,24805,endothelial,pos,45.475912,45,ETS:ELF-ETV#1_45,1,4,new test column
...,...,...,...,...,...,...,...,...,...,...
83,cardiomyocyte-pos.pattern_40,27,cardiomyocyte,pos,29.444444,39,HD:MEIS-TGIF_39,1,83,new test column
84,endothelial-pos.pattern_39,25,endothelial,pos,93.160000,71,HSF1_71,1,84,new test column
85,endothelial-pos.pattern_40,23,endothelial,pos,53.347826,73,Unknown_73,1,85,new test column
86,endothelial-pos.pattern_41,22,endothelial,pos,43.545455,74,SOX#1_74,1,86,new test column


The best way to delete columns is with the `.delete_column()` method, which can take in either a single column name or a list of column names in order to delete them from the metadata.

In [43]:
mc_tutorial.delete_columns("1_value")
mc_tutorial

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster_id,annotation,test_column_2,test_column_3
0,cardiomyocyte-neg.pattern_0,55,cardiomyocyte,neg,114.309091,41,NFYrepressive_41,0,new test column
1,cardiomyocyte-neg.pattern_1,50,cardiomyocyte,neg,81.700000,42,YY1-2repressive_42,1,new test column
2,cardiomyocyte-neg.pattern_2,46,cardiomyocyte,neg,53.630435,43,ZBT7A.H13CORE.0.P.B_43,2,new test column
3,cardiomyocyte-neg.pattern_3,22,cardiomyocyte,neg,46.090909,44,ZEB-SNAI_44,3,new test column
4,endothelial-pos.pattern_0,24805,endothelial,pos,45.475912,45,ETS:ELF-ETV#1_45,4,new test column
...,...,...,...,...,...,...,...,...,...
83,cardiomyocyte-pos.pattern_40,27,cardiomyocyte,pos,29.444444,39,HD:MEIS-TGIF_39,83,new test column
84,endothelial-pos.pattern_39,25,endothelial,pos,93.160000,71,HSF1_71,84,new test column
85,endothelial-pos.pattern_40,23,endothelial,pos,53.347826,73,Unknown_73,85,new test column
86,endothelial-pos.pattern_41,22,endothelial,pos,43.545455,74,SOX#1_74,86,new test column


In [44]:
mc_tutorial.delete_columns(["test_column_2", "test_column_3"])
mc_tutorial

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster_id,annotation
0,cardiomyocyte-neg.pattern_0,55,cardiomyocyte,neg,114.309091,41,NFYrepressive_41
1,cardiomyocyte-neg.pattern_1,50,cardiomyocyte,neg,81.700000,42,YY1-2repressive_42
2,cardiomyocyte-neg.pattern_2,46,cardiomyocyte,neg,53.630435,43,ZBT7A.H13CORE.0.P.B_43
3,cardiomyocyte-neg.pattern_3,22,cardiomyocyte,neg,46.090909,44,ZEB-SNAI_44
4,endothelial-pos.pattern_0,24805,endothelial,pos,45.475912,45,ETS:ELF-ETV#1_45
...,...,...,...,...,...,...,...
83,cardiomyocyte-pos.pattern_40,27,cardiomyocyte,pos,29.444444,39,HD:MEIS-TGIF_39
84,endothelial-pos.pattern_39,25,endothelial,pos,93.160000,71,HSF1_71
85,endothelial-pos.pattern_40,23,endothelial,pos,53.347826,73,Unknown_73
86,endothelial-pos.pattern_41,22,endothelial,pos,43.545455,74,SOX#1_74


## Images

You may have noticed that when printing out a MotifCompendium object, there is a section at the bottom for images. This is because MotifCompendium objects can also store visual metadata about each motif. For instance, it can store forward/reverse logos for each motif, or logos of the closest match in a known database. 

This visual metadata is most naturally populated when performing certain plotting or labeling actions. For instance, when creating a summary table, forward and reverse logos of each motif are generated. When annotating motifs from either an external MotifCompendium or a PFM database, logos of the closest matches are generated.

Image metadata is stored as UTF-8 embeded strings. **Unlike the normal metadata, the image metadata cannot be accessed or modified as directly.**

In many cases, the image metadata will be populated automatically. One way you can manually add images is by calling the `.add_logos()` method, which takes in a motif stack of shape `(N, L, 4)` and saves the logo of each motif in the provided motif stack in the metadata of the MotifCompendium. The `.add_logo()` method also takes in a name for the saved images.

Below, we will add images for the forward/reverse logos of each motif. Notice how we use the `.get_standard_motif_stack()` method in order to pass 4 channel motifs into `.add_logos()`.

In [45]:
mc_tutorial.add_logos(mc_tutorial.get_standard_motif_stack(), "logo (fwd)")
mc_tutorial.add_logos(utils_motif.reverse_complement(mc_tutorial.get_standard_motif_stack()), "logo (rev)")
mc_tutorial

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster_id,annotation
0,cardiomyocyte-neg.pattern_0,55,cardiomyocyte,neg,114.309091,41,NFYrepressive_41
1,cardiomyocyte-neg.pattern_1,50,cardiomyocyte,neg,81.700000,42,YY1-2repressive_42
2,cardiomyocyte-neg.pattern_2,46,cardiomyocyte,neg,53.630435,43,ZBT7A.H13CORE.0.P.B_43
3,cardiomyocyte-neg.pattern_3,22,cardiomyocyte,neg,46.090909,44,ZEB-SNAI_44
4,endothelial-pos.pattern_0,24805,endothelial,pos,45.475912,45,ETS:ELF-ETV#1_45
...,...,...,...,...,...,...,...
83,cardiomyocyte-pos.pattern_40,27,cardiomyocyte,pos,29.444444,39,HD:MEIS-TGIF_39
84,endothelial-pos.pattern_39,25,endothelial,pos,93.160000,71,HSF1_71
85,endothelial-pos.pattern_40,23,endothelial,pos,53.347826,73,Unknown_73
86,endothelial-pos.pattern_41,22,endothelial,pos,43.545455,74,SOX#1_74


The `.add_logos()` method also takes in an optional argument called `trim`. `trim` can either be `True`/`False` or a number between 0 and 1, inclusive. `trim` dictates of the flanks of each motif are trimmed off when creating a logo for the motif. This can be useful if your motif has large empty or uninformative flanks that you don't want to be a part of your logo. If `trim` is False, then no trimming occurs. If `trim` is True, then motifs are trimmed at a standard threshold of `1/L`. If a number is provided, then trimming is done so that flank positions with importance less than `trim` are not shown. For example, when setting `trim=0`, all flank positions with 0 importance are removed. By default, `trim` is set to `False` and no trimming is done.

In [46]:
mc_tutorial.add_logos(mc_tutorial.get_standard_motif_stack(), "logo (fwd) standard trim", trim=True)
mc_tutorial.add_logos(mc_tutorial.get_standard_motif_stack(), "logo (fwd) zero trim", trim=0)
mc_tutorial.add_logos(mc_tutorial.get_standard_motif_stack(), "logo (fwd) 0.15 trim", trim=0.15)

To get a list of all saved images, use the `.images()` method.

In [47]:
mc_tutorial.images()

['logo (fwd)',
 'logo (rev)',
 'logo (fwd) standard trim',
 'logo (fwd) zero trim',
 'logo (fwd) 0.15 trim']

If you would like to get the UTF-8 embedded image strings of a particular image, you can use the `.get_images()` method and pass it the name of the saved images you want to retrieve. As you can see, these strings are quite arduous to work with so it is not recommended to ever interact with them yourself.

In [48]:
fwd_logo_strings = mc_tutorial.get_images("logo (fwd)")
print(len(fwd_logo_strings), type(fwd_logo_strings))
fwd_logo_strings[0]

88 <class 'list'>


'iVBORw0KGgoAAAANSUhEUgAAAeQAAACuCAYAAADnGn5HAAAAOnRFWHRTb2Z0d2FyZQBNYXRwbG90bGliIHZlcnNpb24zLjEwLjAsIGh0dHBzOi8vbWF0cGxvdGxpYi5vcmcvlHJYcgAAAAlwSFlzAAAPYQAAD2EBqD+naQAALFlJREFUeJzt3Xl8HNWV6PFfVfWq1ZIla7Elg228G4zBARwI2CxJWMOWBUIICckkhOENGUiAQCYZ4L1skDdhSRiSQAIhZAJMAsmwGSdsD8eYzcZgsPGCLdmytW+9V70/rtXqrq6WuiW11S2d7+fTH6mqq7pvS911+px765ZmWZaFEEIIIcaVPt4NEEIIIYQEZCGEECIvSEAWQggh8oAEZCGEECIPSEAWQggh8oAEZCGEECIPSEAWQggh8oAEZCGEECIPSEAWQggh8oBrvBsghBAFq+mvsOcp6H4XQq1QMhur8mgsz1TQfRDuhFALAFrnW6CBphlQexrM/cb4tl3kHS3TqTP/9YrLCXXupcgVw6Vb9IY9BKIQtcC0LNyGiceIoWFR4gljoWFZGm0BL/uCOgF/Dz1F3fT6u4lpJt6wH1/ER3GgBEuz6PZ3E3VH6Hf34415KA2UUBYoxxssJhyqIBisIBbzE4u5AR2PpwvDCKHrUWDgJVho2oHfLA1Ni+HztxHVo0RcESJGhJgew9RMNEvDE/NgYRFxRQ78MTRcMRfumBt3zI3LdGHESnBbZbgsFxouwjE3pt4LegBNj+DSY7iNKKDhc6mfMdOgP+Ki1+qnO1JKPy4CmkVU03Bj4nUF8EV1NM0k5A4T1aPEjCiGqeON+PBGfBRrEVxGFA0DNB00iJg6phZG1yw0ywLdRNdNsMCtgYaGZmlYpotg1KCl5WiCwSrC4VJiMS8+XzslJbtwuYLoeuTA32vw329ZGqDj97cQiZTS0TGPUKiCSKQI03RRUfEeHk83hhFG00yH/XXAoqJiC2EjTMAVpN+AkAY

The best way to rename saved images is with the `.rename_images()` method, which takes in a dictionary of old saved image names to new saved image names.

In [49]:
mc_tutorial.rename_images({"logo (fwd)": "logo (forward)"})
mc_tutorial

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster_id,annotation
0,cardiomyocyte-neg.pattern_0,55,cardiomyocyte,neg,114.309091,41,NFYrepressive_41
1,cardiomyocyte-neg.pattern_1,50,cardiomyocyte,neg,81.700000,42,YY1-2repressive_42
2,cardiomyocyte-neg.pattern_2,46,cardiomyocyte,neg,53.630435,43,ZBT7A.H13CORE.0.P.B_43
3,cardiomyocyte-neg.pattern_3,22,cardiomyocyte,neg,46.090909,44,ZEB-SNAI_44
4,endothelial-pos.pattern_0,24805,endothelial,pos,45.475912,45,ETS:ELF-ETV#1_45
...,...,...,...,...,...,...,...
83,cardiomyocyte-pos.pattern_40,27,cardiomyocyte,pos,29.444444,39,HD:MEIS-TGIF_39
84,endothelial-pos.pattern_39,25,endothelial,pos,93.160000,71,HSF1_71
85,endothelial-pos.pattern_40,23,endothelial,pos,53.347826,73,Unknown_73
86,endothelial-pos.pattern_41,22,endothelial,pos,43.545455,74,SOX#1_74


UTF-8 embedded images can take up quite a bit of memory so it is recommended to delete images that you won't regularly use or need. The best way to delete saved images is with the `.delete_images()` method, which can take as input a saved image or a list of saved images to delete.

In [50]:
mc_tutorial.delete_images(["logo (forward)", "logo (rev)"])
mc_tutorial

,name,num_seqlets,model,posneg,avg_dist_from_summit,cluster_id,annotation
0,cardiomyocyte-neg.pattern_0,55,cardiomyocyte,neg,114.309091,41,NFYrepressive_41
1,cardiomyocyte-neg.pattern_1,50,cardiomyocyte,neg,81.700000,42,YY1-2repressive_42
2,cardiomyocyte-neg.pattern_2,46,cardiomyocyte,neg,53.630435,43,ZBT7A.H13CORE.0.P.B_43
3,cardiomyocyte-neg.pattern_3,22,cardiomyocyte,neg,46.090909,44,ZEB-SNAI_44
4,endothelial-pos.pattern_0,24805,endothelial,pos,45.475912,45,ETS:ELF-ETV#1_45
...,...,...,...,...,...,...,...
83,cardiomyocyte-pos.pattern_40,27,cardiomyocyte,pos,29.444444,39,HD:MEIS-TGIF_39
84,endothelial-pos.pattern_39,25,endothelial,pos,93.160000,71,HSF1_71
85,endothelial-pos.pattern_40,23,endothelial,pos,53.347826,73,Unknown_73
86,endothelial-pos.pattern_41,22,endothelial,pos,43.545455,74,SOX#1_74


Please refer to [Tutorial 4](4_visualization.ipynb) to learn more about how to visualize and make use of these images.

## Miscellaneous

The length of a MotifCompendium object will return the number of motifs in the MotifCompendium.

In [51]:
len(mc_tutorial)

88

MotifCompendum objects can be printed directly.

In [52]:
print(mc_tutorial)

MotifCompendium with 88 motifs.
--- Motifs = (88, 30, 8) ---

--- Metadata ---
                            name  num_seqlets          model posneg  \
0    cardiomyocyte-neg.pattern_0           55  cardiomyocyte    neg   
1    cardiomyocyte-neg.pattern_1           50  cardiomyocyte    neg   
2    cardiomyocyte-neg.pattern_2           46  cardiomyocyte    neg   
3    cardiomyocyte-neg.pattern_3           22  cardiomyocyte    neg   
4      endothelial-pos.pattern_0        24805    endothelial    pos   
..                           ...          ...            ...    ...   
83  cardiomyocyte-pos.pattern_40           27  cardiomyocyte    pos   
84    endothelial-pos.pattern_39           25    endothelial    pos   
85    endothelial-pos.pattern_40           23    endothelial    pos   
86    endothelial-pos.pattern_41           22    endothelial    pos   
87    endothelial-pos.pattern_42           21    endothelial    pos   

    avg_dist_from_summit  cluster_id              annotation  
0    

# Part 4 - Conclusion

This tutorial described all the ways in which you can interact with and manipulate MotifCompendium objects and their internal parameters. Hopefully, this will help you work with MotifCompendium objects in a more flexible and object-oriented way.

Thank you so much for trying out MotifCompendium!